# 第 8 章 · 动态工作流基础：用 Python 编排 Agent

> **你已会**：多智能体（第 5 章）、State（第 6 章）。
>
> **读完后你能**：用 `Workflow + @node + ctx.run_node()` 写出顺序、传值、条件、有界迭代和有界并发。
>
> **本章不碰**：暂停、恢复、重试、幂等。那是第 9 章。

**怎么读**

- **主线（约 45 分钟）**：§1 概念 → §2 最小骨架 → §3 流水线 → §4 条件与迭代。
- **深读**：§5 并发、§6 与 LangGraph 的对照。
- 环境：`uv sync`，再用 `uv run --env-file .env jupyter lab` 启动。本章真实调用 DeepSeek **4 次**。

---

## 1. 从第五章迁移：只换了编排接口

| 第五章 | 本章 | 变化 |
|---|---|---|
| `SequentialAgent(sub_agents=[A, B, C])` | 依次 `await ctx.run_node(A/B/C, input)` | 顺序由 Python 表达 |
| `output_key` + `{key}` 插值 | 返回值 + 显式输入 | 局部数据不必经过 State |
| `ParallelAgent` | parallel worker / `asyncio.gather` | 批次数量可由输入决定 |
| `LoopAgent` + `escalate` | 有上限的 Python 循环 | 退出条件和失败结果直接可见 |

一句话：**第五章把"走哪几步"写成了一份数据（节点列表），本章把它写成了一段程序（Python 函数体）。**

### 1.1 "动态"到底指什么

这个词在 Agent 领域至少有三层意思：

| 「动态」可能指 | 谁决定路径 | 在 ADK 里对应什么 | 本章是它吗 |
|---|---|---|---|
| ① 模型自主规划 | LLM 临场推理 | 第 5 章的 `transfer_to_agent` | **不是** |
| ② 运行时创建 Agent | 系统边跑边生成 | 一些多智能体框架的"动态组队" | **不是** |
| ③ 用代码表达路径 | 你写的 Python | `Workflow` + `@node` + `ctx.run_node()` | ✅ **就是它** |

> 📌 **本章的"动态"不是"AI 自己做主"，而是"控制权在你手里"。**

理解这一点，关键要看它的对立面。**"动态"的对立面不是"图"，而是"预先声明的边"。**

本章的主流水线只声明了一条边：

```python
pipeline = Workflow(name="pipeline", edges=[("START", pipeline_entry)])
```

声明图上只有一个入口节点。真正的三步写作、条件分支、有界循环，全在 `pipeline_entry` / `writing_pipeline` 的函数体里。所以打开 `adk web` 的 Graph 面板看它，几乎画不出什么——**流程不在图里，在代码里。**

<details>
<summary><b>深入：官方定义与 graph-scheduled nodes</b></summary>

ADK 源码在 `max_concurrency` 字段上写着：

> **Dynamic nodes (via `ctx.run_node()`) are excluded** —— 它们由父节点内联等待。

于是"动态节点"的定义是：**父节点在运行时通过 `ctx.run_node()` 调起来的节点**。与之相对的是 **graph-scheduled nodes**：由 `edges` 声明、构造时编译成图并被校验的节点。

本章的 `Workflow` 仍然是图工作流（官方文档把 *Dynamic workflows* 放在 *Graph Workflows* 大类下），只是边不由你声明，而由代码走出来。

</details>

### 1.2 两个独立的维度

不要把它们混为一谈：

- **图还是代码** → 决定**路径如何表达**（本章选"代码"）；
- **规则还是 LLM** → 决定**谁作决策**（本章的检查用规则，写作交给 LLM）。

两者可以自由组合。本章落在"代码 + 规则"这一格，用的是 `review["passed"]` 这样的规则字段。你也可以写"代码 + LLM"：让 `ctx.run_node()` 调起的 Agent 在自己内部跑 ReAct 循环。

> 📌 **区别在层级**：Agent 内部的 ReAct 只决定"这一步内部怎么做"，不决定"下一步是谁"。

<details>
<summary><b>深入：为什么 Workflow 不能当 sub_agents</b></summary>

在 ADK 2.x 中（2.12 实测未变），`Workflow` 不能直接充当 `LlmAgent.sub_agents`。原因在类型继承：`BaseAgent` 继承 `BaseNode`，而 `Workflow` 只继承 `BaseNode`。

**它是"能被调度的一步"，但不是"有自主性的智能体"。**

</details>

### 1.3 这个选择的三个后果

这些不是 Bug，而是设计的必然：

1. **跑之前画不出流程图。** `node_info.path` 是执行时拼出来的（如 `hello@1/hello_entry@1`），不是谁声明的。
2. **并发上限只能挂在节点上。** 动态节点由父节点内联 `await`，工作流不知道同时有几个在跑；再对它们限流会造成死锁。
3. **恢复不能"从函数中间继续"。** Python 无法从函数体某一行接着执行，所以第 9 章要靠"父函数重入 + 子节点回放"。

还有一条代价：静态图能在**构造时**报出环、不可达节点、schema 不匹配；**动态工作流只能等跑起来才知道**。所以本章和第 9 章都要自己立契约、自己写断言。

```mermaid
flowchart LR
    I[主题] --> O[大纲 Agent]
    O --> W[写作 Agent]
    W --> P[润色 Agent]
    P --> C[Python 检查]
    C -->|通过| F[交付]
    C -->|未通过且有预算| R[修改 Agent]
    R --> C
```

> 📌 这张图**不是** `edges` 声明出来的，而是**函数体走出来的路径**——画给人看的，不是给框架看的。

## 2. 最小运行骨架：先认识三个构件

- `@node`：把函数登记为框架节点。
- `ctx.run_node()`：执行子节点并取得返回值。
- `Workflow(edges=[("START", entry)])`：把 Runner 入口接到编排函数。

**先猜：这个 `Workflow` 声明了几条边？** 运行后看输出，再回来看 §1.1。

调用子节点的父函数设置 `rerun_on_resume=True`；普通叶子节点保持默认值。默认参数绑定中，`ctx` 和 `node_input` 是特殊接口名，其他参数名会从 State 取值。

> 📌 **`@node` 不只是"贴个装饰器"**，它是你和框架之间的**登记动作**：只有登记过的函数，框架才认识它、才会在 checkpoint 里保存它的结果。

### 2.1 `rerun_on_resume`：先记住结论

完整机制在第 9 章展开。现在只需要理解到"够用"：

- 一个节点如果内部调用了别的节点（里面有 `ctx.run_node(...)`），它就是"父函数"。
- 恢复时，框架**不是**从断点"续上"，而是**重新进入父函数、从头再跑一遍**。
- 但已经成功完成的子节点**不会重新执行**——结果从 checkpoint 直接还给父函数。
- 所以父函数的普通代码必须能重复执行；**会改变外部世界的动作要放进叶子节点**。

> ⚠️ 这里先记住结论。第 9 章会用执行计数证明父函数确实重入了、子节点确实只执行了一次。

In [1]:
import asyncio
import importlib.metadata as metadata
import json
import os
import sys
from uuid import uuid4
from dotenv import load_dotenv

load_dotenv()

# 使用随 LiteLLM 安装的成本表，不影响真实模型请求。
os.environ["LITELLM_LOCAL_MODEL_COST_MAP"] = "True"

from google.adk import Context, Workflow
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.workflow import node
from google.genai import types
from pydantic import BaseModel, Field

assert os.getenv("DEEPSEEK_API_KEY"), "请先配置 DEEPSEEK_API_KEY，再执行本章。"
_adk_version = metadata.version("google-adk")
assert _adk_version.startswith("2.12."), (
    f"当前内核里的 google-adk 是 {_adk_version}（解释器：{sys.executable}），本章需要 2.12.x。\n"
    "请从仓库根目录用 `uv run --env-file .env jupyter lab` 启动，并确认右上角内核来自项目 .venv；"
    "换过依赖后要 Restart Kernel。"
)
APP, USER = "adk_ch07", "student"
checks = {}
print("运行环境：", _adk_version, "；DeepSeek 密钥已配置")

运行环境： 2.12.0 ；DeepSeek 密钥已配置


Runner 接收 `types.Content`。下面这三个辅助函数只做基础工作，先不加入日志、重试或恢复协议。

**为什么不能只用 `is_final_response()` 取流程结果？** 因为子节点也会产生最终事件。`node_info.output_for` 记录了输出的归属路径；**不含 `/` 的路径**指向根 Workflow，例如 `hello@1`。

> 📌 三个函数各负责一件事：**归一入口**（`input_text`）、**按归属取根输出**（`collect`）、**把"建会话 → 跑一次"打包成一个 `await`**（`start`）。

> 📌 `hello@1` 的读法：`hello` 是工作流名，`@1` 是本次执行里的调用序号。同一个节点被调第二次就是 `@2`。路径带序号，正是因为"路径是执行出来的"。

In [2]:
def input_text(value) -> str:
    if isinstance(value, types.Content):
        return "".join(part.text or "" for part in value.parts or [])
    if isinstance(value, str):
        return value
    raise TypeError(f"入口需要 Content 或 str，实际是 {type(value).__name__}")


async def collect(runner, session_id, message, *, invocation_id=None):
    output, events = None, []
    async for event in runner.run_async(
        user_id=USER,
        session_id=session_id,
        new_message=message,
        invocation_id=invocation_id,
    ):
        events.append(event)
        if event.node_info and event.output is not None:
            if any(
                "/" not in path for path in event.node_info.output_for or []
            ):
                output = event.output
    return output, events


async def start(
    workflow, payload, *, session_service=None, artifact_service=None
):
    service = session_service or InMemorySessionService()
    session_id = uuid4().hex
    await service.create_session(
        app_name=APP, user_id=USER, session_id=session_id
    )
    runner = Runner(
        agent=workflow,
        app_name=APP,
        session_service=service,
        artifact_service=artifact_service,
    )
    text = (
        payload
        if isinstance(payload, str)
        else json.dumps(payload, ensure_ascii=False)
    )
    message = types.Content(role="user", parts=[types.Part(text=text)])
    output, events = await collect(runner, session_id, message)
    return output, events, runner, session_id

In [3]:
@node
def normalize_topic(node_input: str) -> str:
    return node_input.strip().removeprefix("主题：")


@node(rerun_on_resume=True)
async def hello_entry(ctx: Context, node_input):
    topic = await ctx.run_node(normalize_topic, input_text(node_input))
    return f"学习主题：{topic}"


hello = Workflow(name="hello", edges=[("START", hello_entry)])
result, events, *_ = await start(hello, "  主题：动态工作流  ")
assert result == "学习主题：动态工作流"
checks["最小工作流与根输出"] = True
for event in events:
    if event.node_info and event.output is not None:
        print(
            event.node_info.path,
            "→",
            event.node_info.output_for,
            "→",
            event.output,
        )

hello@1/hello_entry@1/normalize_topic@1 → ['hello@1/hello_entry@1/normalize_topic@1'] → 动态工作流
hello@1/hello_entry@1 → ['hello@1/hello_entry@1', 'hello@1'] → 学习主题：动态工作流


## 3. 真实 DeepSeek 流水线：大纲 → 初稿 → 润色

继续用第 5 章的主题"程序员为什么需要学习写作"。这次不写 `output_key`，而是把上一步结果**显式传给下一步**。

**先猜：这三次模型调用，是框架自动串起来的，还是 Python 代码决定的？**

DeepSeek 的 JSON 方案沿用第 3 章：`response_format={"type": "json_object"}` + 本地 Pydantic 校验。不设置 `output_schema`，避免请求被转换成 DeepSeek 不兼容的 `json_schema`。

> 📌 **边读边对照三件事**：
>
> 1. **Python 决定顺序**：三次 `ctx.run_node` 依次写下来；
> 2. **模型只负责单步**：每个 Agent 只做"列要点 / 写正文 / 润色"中的一件；
> 3. **数据显式流动**：上一步的返回值当下一步的入参，不经过 State。

> ⚠️ **JSON 的三层保证**：格式靠 `response_format`，字段靠 Pydantic 校验，业务要求还要另外检查（§4）。

In [4]:
model_responses = []


def record_response(callback_context, llm_response):
    usage = llm_response.usage_metadata
    model_responses.append(
        {
            "agent": callback_context.agent_name,
            "tokens": usage.total_token_count if usage else None,
        }
    )
    return None


def deepseek():
    return LiteLlm(
        model="deepseek/deepseek-chat",
        max_tokens=768,
        response_format={"type": "json_object"},
    )


class Outline(BaseModel):
    points: list[str] = Field(min_length=3, max_length=3)


class Draft(BaseModel):
    title: str = Field(min_length=1)
    body: str = Field(min_length=1)


outliner = Agent(
    name="outliner",
    model=deepseek(),
    mode="single_turn",
    instruction="根据输入 topic 和 notes 列出三个写作要点，遵循 notes 的事实。"
    '只输出 JSON：{"points": ["要点1", "要点2", "要点3"]}。',
    after_model_callback=record_response,
)
writer = Agent(
    name="writer",
    model=deepseek(),
    mode="single_turn",
    instruction="根据 topic、notes 和 outline 写中文入门短文。正文目标 80～100 个字符，"
    "验收范围 60～120 个字符，标点和英文字母也计数。只解释核心机制，"
    "使用 required_terms 中的每个词，包含以‘例如’引出的实际例子。"
    "例子使用内容写作或审批场景，不引入支付等额外业务。"
    '遵循 notes，只输出 JSON：{"title": "标题", "body": "正文"}。',
    after_model_callback=record_response,
)
polisher = Agent(
    name="polisher",
    model=deepseek(),
    mode="single_turn",
    instruction="你是保守的中文编辑。润色输入 draft；若有 findings，只修复所列问题，"
    "已经满足的规则不要破坏。长度已合格时保持原长度，不要压缩。遵循 request.notes，"
    "正文目标 80～100 个字符，验收范围 60～120，标点和英文字母也计数。"
    "正文写成四句完整句子，每句约 20～25 字符：核心观点、作用说明、"
    "例如场景、实际效果。过短的稿件必须扩写，不能只替换字词或压成两句。"
    "包含 request.required_terms 和‘例如’实际例子。"
    "例子限于内容写作或审批，不引入支付等额外业务。"
    '保留准确含义，只输出 JSON：{"title": "标题", "body": "正文"}。',
    after_model_callback=record_response,
)

`mode="single_turn"` 是**声明意图，不是必需**：Agent 作为 Workflow 节点被 `ctx.run_node()` 调用时，默认模式本来就是 `single_turn`。

| 取值 | 含义 | 典型场景 |
|---|---|---|
| `chat` | 可被 `transfer_to_agent` 转交的对话 Agent | 子 Agent、需要互相转交 |
| `task` | 与用户多轮对话来完成任务 | 需要追问澄清 |
| `single_turn` | 完成一次任务，不与用户对话 | **Workflow 里的节点** |

> 📌 **`chat` 和 `single_turn` 的真正差别不是"参不参与对话"，而是输入从哪来**：
>
> | 模式 | 输入来源 |
> |---|---|
> | `chat` | 会话历史（session events） |
> | `single_turn` | 上游节点直接传来的 `node_input` |

所以把 `chat` 模式的 Agent 接在别的节点后面，ADK 在**构造工作流图时就会拒绝**——它拿不到上游数据。

<details>
<summary><b>完整报错长什么样</b></summary>

```text
The agent '...' has been added to the workflow with mode='chat' following node '...'.
This is not supported because chat-mode agents rely on conversational history
(session events) and cannot consume direct node inputs from preceding nodes.
Please change the agent's mode to 'single_turn'
```

</details>

本章显式写出 `mode="single_turn"`，是为了把"这个 Agent 吃节点输入、不吃聊天记录"写在脸上。参数全貌见第 3 章。

In [5]:
@node(rerun_on_resume=True)
async def writing_pipeline(ctx: Context, node_input: dict) -> dict:
    outline = Outline.model_validate_json(
        await ctx.run_node(outliner, node_input)
    )
    draft = Draft.model_validate_json(
        await ctx.run_node(
            writer,
            {**node_input, "outline": outline.model_dump()},
        )
    )
    final = Draft.model_validate_json(
        await ctx.run_node(
            polisher,
            {"request": node_input, "draft": draft.model_dump()},
        )
    )
    return {"outline": outline.model_dump(), "draft": final.model_dump()}


@node(rerun_on_resume=True)
async def pipeline_entry(ctx: Context, node_input):
    return await ctx.run_node(
        writing_pipeline, json.loads(input_text(node_input))
    )


pipeline = Workflow(name="pipeline", edges=[("START", pipeline_entry)])
writing_request = {
    "topic": "程序员为什么需要学习写作",
    "notes": "写作有助于澄清需求、记录设计理由、减少团队沟通成本。",
    "required_terms": ["例如"],
}
before = len(model_responses)
pipeline_result, pipeline_events, *_ = await start(pipeline, writing_request)
assert len(model_responses) - before == 3
assert [row["agent"] for row in model_responses[before:]] == [
    "outliner",
    "writer",
    "polisher",
]
agent_paths = [
    event.node_info.path
    for event in pipeline_events
    if event.node_info
    and event.content
    and event.author in {"outliner", "writer", "polisher"}
]
checks["真实模型流水线与执行顺序"] = True
print(json.dumps(pipeline_result, ensure_ascii=False, indent=2))
print("实际模型调用路径：", *agent_paths, sep="\n")

{
  "outline": {
    "points": [
      "写作能帮助程序员澄清需求，例如把模糊的用户诉求整理成明确的验收标准",
      "写作能记录设计理由，例如在技术方案文档中说明为何选择某种架构而非其他方案",
      "写作能减少团队沟通成本，例如通过清晰的文档让成员异步获取信息，避免重复解释"
    ]
  },
  "draft": {
    "title": "程序员为何要学写作",
    "body": "写作能帮程序员澄清需求。例如把模糊诉求写成明确验收标准。也便于记录设计理由与选型依据。还能降低团队沟通成本，提升协作效率。"
  }
}
实际模型调用路径：
pipeline@1/pipeline_entry@1/writing_pipeline@1/outliner@1
pipeline@1/pipeline_entry@1/writing_pipeline@1/writer@1
pipeline@1/pipeline_entry@1/writing_pipeline@1/polisher@1


> 📌 **两种控制转移**：`ctx.run_node()` 是**调用—返回**，执行完把控制权还给你；`transfer_to_agent` 是**移交—接管**，交出去就不回来了。

> ⚠️ **返回类型不要混**：函数节点返回 Pydantic 模型时会变成**字典**；Agent 节点在没设置 `output_schema` 时返回的是 **JSON 字符串**。把两者当同一种东西处理，是本章最常见的报错来源。

State 仍适合工具/回调共享的摘要、跨步骤流程状态和 instruction 插值。**普通中间结果优先显式传值**，恢复原理在第 9 章。

---

## 4. 条件与有界迭代：检查不通过，就修改

继续同一写作场景。业务规则检查标题、正文长度、必要术语和实际例子的标记。

**先猜：检查不通过时，流程会抛异常，还是返回一个普通结果？**

> 📌 **业务不合格 ≠ 程序出错。** `passed=False` 是一个**业务结论**，它应该走进修改分支，而不是触发网络重试或抛异常。把"业务判断"和"运行故障"分开，是让流程可预期的关键。

`max_revisions` 表示最多修改几次，初次检查不占修改预算。达到上限必须返回 `revision_limit`，**不能把最后一稿自动标成成功**。

In [6]:
class WritingRequest(BaseModel):
    topic: str = Field(min_length=1)
    notes: str = Field(min_length=1)
    required_terms: list[str] = Field(default_factory=list)


@node
def review_article(node_input: dict) -> dict:
    request = WritingRequest.model_validate(node_input["request"])
    draft = Draft.model_validate(node_input["draft"])
    findings = []
    if not draft.title.strip():
        findings.append("标题不能为空白")
    if not 60 <= len(draft.body) <= 120:
        findings.append(f"正文 {len(draft.body)} 字，要求 60～120 字")
    for term in dict.fromkeys(["例如", *request.required_terms]):
        if term not in draft.body:
            findings.append(f"正文必须包含：{term}")
    return {"passed": not findings, "findings": findings}


@node(rerun_on_resume=True)
async def refine_article(ctx: Context, node_input: dict) -> dict:
    request, draft = node_input["request"], node_input["draft"]
    budget = node_input["max_revisions"]
    if not isinstance(budget, int) or isinstance(budget, bool) or budget < 0:
        raise ValueError("max_revisions 必须是非负整数")
    for revision in range(budget + 1):
        review = await ctx.run_node(
            review_article, {"request": request, "draft": draft}
        )
        if review["passed"] or revision == budget:
            return {
                "draft": draft,
                **review,
                "revisions": revision,
                "stop_reason": "passed"
                if review["passed"]
                else "revision_limit",
            }
        draft = Draft.model_validate_json(
            await ctx.run_node(
                polisher,
                {
                    "request": request,
                    "draft": draft,
                    "findings": review["findings"],
                },
            )
        ).model_dump()


@node(rerun_on_resume=True)
async def refinement_entry(ctx: Context, node_input):
    return await ctx.run_node(
        refine_article, json.loads(input_text(node_input))
    )


refinement = Workflow(name="refinement", edges=[("START", refinement_entry)])

In [7]:
valid_draft = {
    "title": "程序员的写作练习",
    "body": "写作可以帮助程序员梳理需求、说明设计理由并减少沟通误差。例如，在接口变更前写清输入、输出与兼容范围，团队就能及时发现遗漏，也方便后来维护的人理解当时的选择。",
}
missing_example = {
    **valid_draft,
    "body": valid_draft["body"].replace("例如", "比如"),
}
cases = [
    (valid_draft, 0, True),
    ({"title": "初稿", "body": "太短"}, 0, False),
    (missing_example, 1, True),
]
for draft, budget, expected in cases:
    result, *_ = await start(
        refinement,
        {
            "request": writing_request,
            "draft": draft,
            "max_revisions": budget,
        },
    )
    print(json.dumps(result, ensure_ascii=False))
    assert result["passed"] is expected, result
    assert result["revisions"] <= budget
checks["一次通过、预算耗尽与真实模型修改"] = True

{"draft": {"title": "程序员的写作练习", "body": "写作可以帮助程序员梳理需求、说明设计理由并减少沟通误差。例如，在接口变更前写清输入、输出与兼容范围，团队就能及时发现遗漏，也方便后来维护的人理解当时的选择。"}, "passed": true, "findings": [], "revisions": 0, "stop_reason": "passed"}
{"draft": {"title": "初稿", "body": "太短"}, "passed": false, "findings": ["正文 2 字，要求 60～120 字", "正文必须包含：例如"], "revisions": 0, "stop_reason": "revision_limit"}


{"draft": {"title": "程序员的写作练习", "body": "写作是程序员梳理思路的重要工具。它能帮助澄清需求并记录设计理由。例如写接口变更说明时，列清输入输出与兼容范围。这样能减少沟通误差，也方便后来者维护。"}, "passed": true, "findings": [], "revisions": 1, "stop_reason": "passed"}


第三次运行把已合格正文中的"例如"改成"比如"，制造一个明确缺陷，真实调用 DeepSeek 修复，再重新检查；前两次不调用模型。这样能分别观察**编排规则**和**模型行为**。

修改只应修复问题，不能破坏已经合格的长度。

### 4.1 为什么提示词里的目标要比验收范围更严

提示词写的是 **80～100 字符**，代码验收用的是 **60～120 字符**。两个数字故意不一样：

- 模型的输出长度**天生有抖动**。把验收线直接写进提示词，模型会贴着边界输出，一半样本掉到范围外，验收就会**随机失败**。
- 提示词给更窄的目标区间，验收用更宽的硬边界，中间留出抖动余量。

> 💡 **如果模型仍然不达标？** 本章的验收会**如实失败**：`stop_reason` 返回 `revision_limit`，最后一稿不会被标成成功。这不是 Bug，是设计。处理顺序：① 收紧提示词目标；② 提高 `max_revisions`；③ 放宽到业务真正在意的口径。**不要**为了让 Notebook 变绿就把检查改宽到总是通过。

标点和英文字母按 Python `len()` 计数。

> ⚠️ **检查的边界**：这些规则只证明标题、长度和术语符合要求，**不是事实评估**。"包含 Python"不等于"准确理解 Dynamic workflows"。第 6 节会给模型提供已核对的事实，并展示全文供人工审核。

---

## 5. 动态批处理：验证并发上限，而不只看耗时

任务数由输入列表决定。`node(worker, parallel_worker=True, max_parallel_workers=2)` 最多同时运行两个工作节点，结果按输入顺序返回。

**先猜：同时提交 10 个任务，峰值并发由什么决定？总耗时能证明并发发生了吗？**

> 📌 这里用 `node(...)` 再包一层，是为了挂上 `parallel_worker` 这类**节点级配置**。
>
> **为什么并发上限挂在节点上，而不是工作流上？** 因为动态节点由父节点内联 `await`，工作流不知道同时有几个在跑；框架的 `max_concurrency` 明确排除动态节点。对已经在等的父节点限流会造成死锁。**并发控制只能落在被调的那个节点上。**

In [8]:
concurrency = {"active": 0, "peak": 0}


@node
async def inspect_topic(node_input: str) -> dict:
    concurrency["active"] += 1
    concurrency["peak"] = max(concurrency["peak"], concurrency["active"])
    try:
        await asyncio.sleep(0.03)
        return {"topic": node_input, "characters": len(node_input)}
    finally:
        concurrency["active"] -= 1


bounded_inspector = node(
    inspect_topic, parallel_worker=True, max_parallel_workers=2
)


@node(rerun_on_resume=True)
async def inspect_batch(ctx: Context, node_input):
    return await ctx.run_node(
        bounded_inspector, json.loads(input_text(node_input))
    )


topics = ["节点", "循环", "checkpoint", "人工确认", "状态"]
inspector = Workflow(name="inspector", edges=[("START", inspect_batch)])
concurrency.update(active=0, peak=0)
result, *_ = await start(inspector, topics)
assert concurrency == {"active": 0, "peak": 2}
assert [item["topic"] for item in result] == topics
checks["并发确实发生且不超过上限"] = True
print(
    "实测并发峰值：",
    concurrency["peak"],
    "；返回顺序：",
    [r["topic"] for r in result],
)

实测并发峰值： 2 ；返回顺序： ['节点', '循环', 'checkpoint', '人工确认', '状态']


不同节点并行时，也可以用 `await asyncio.gather(ctx.run_node(A, ...), ctx.run_node(B, ...))`。

> ⚠️ **两个常见误解**：
>
> ① **逐项 `await` 就是串行**——写了 `async` 不等于自动并行；
> ② **CPU 密集任务不会因为加了 `async` 就并行**——`asyncio` 是单线程协作式调度，它让"等 I/O"的时间重叠，不让"算"的时间重叠。

本章的策略是"执行异常让整批失败；业务不合格返回检查结果"。并发分支各自返回结果，由父节点汇总；写同一个 State key 不会自动合并。不要用 `except BaseException` 或 `gather(return_exceptions=True)` 吞掉人工暂停信号。

> 📌 `max_parallel_workers` 只约束**本批次**，**不等于你的 API 配额**。批次嵌套时上限会叠乘，生产环境要另做全局限流。

<details>
<summary><b>深入：与 LangGraph 对照</b></summary>

| 本章构件 | LangGraph 对照 | 差异 |
|---|---|---|
| `Workflow(edges=...)` | Graph API 的 nodes / edges | 路由与状态协议不同 |
| 编排函数 + `ctx.run_node()` | Functional API 的 `entrypoint` / `task` | 都用代码表达控制流；恢复协议不同 |
| 返回值与 `ctx.state` | 任务返回值与 State | **并发写入不自动套用 reducer** |
| `parallel_worker` | Map / 并行任务 | 上限只约束本批次 |

对照阅读：[04-LangGraph核心-图与状态](../langchain-langgraph/04-LangGraph核心-图与状态.ipynb) 的 §3 Reducer 和 §4 条件边。

参考：[Dynamic workflows](https://adk.dev/graphs/dynamic/) · [Data handling](https://adk.dev/graphs/data-handling/) · [DeepSeek JSON Output](https://api-docs.deepseek.com/guides/json_mode/)

</details>

---

## 📌 用五句话复述本章

1. **"动态"= 用 Python 表达路径**；它的对立面是"预先声明的边"，不是"图"。
2. **三个构件**：`@node` 登记、`ctx.run_node()` 调用取值、`Workflow(edges=[("START", entry)])` 接入口。
3. **普通中间结果显式传值**；State 只留给工具/回调共享、跨步骤状态和 instruction 插值。
4. **业务不合格不等于运行异常**；预算耗尽必须如实返回 `revision_limit`。
5. **并发要用数据证明**：记录峰值验证"确实并发且不超上限"，只看耗时证明不了。

---

## 🧪 本章练习

### 1. 给最小骨架加一个参数（基础）

- [ ] 让 `hello` 接收 `{"topic": "...", "audience": "..."}`
- [ ] 在 `hello_entry` 里解包后传给 `normalize_topic`
- [ ] 返回一个包含两个字段的字典
- [ ] 断言根输出同时含有主题和读者
- [ ] 打印所有事件的 `node_info.path` 与 `node_info.output_for`

### 2. 读懂 `output_for` 与根输出（基础）

- [ ] 逐条打印每个事件的 `path`、`output_for` 和 `output`
- [ ] 解释为什么子节点也会产生"最终事件"
- [ ] 解释为什么"不含 `/`"的路径指向根 Workflow

### 3. 把 State 传值改成显式传值（进阶）

- [ ] 把第 5 章的 `output_key` 流水线改写成返回值传值
- [ ] 列出哪些数据不再需要进 State
- [ ] 说明哪一类数据仍然必须留在 State
- [ ] 确认模型调用次数没有增加

### 4. 修订预算的三条路径（进阶）

- [ ] 覆盖一次通过、修改一次后通过、预算耗尽
- [ ] 断言三条路径的 `stop_reason` 与 `revisions`
- [ ] 预算耗尽必须返回失败，不得标为成功
- [ ] 断言哪条路径调用了模型、调用了几次

### 5. 用数据证明并发上限（工程）

- [ ] 批量 20 项，`max_parallel_workers=3`，记录峰值、总耗时、返回顺序
- [ ] 把上限改成 1 和 20 各跑一次
- [ ] 峰值分别不超过 3 / 1 / 20
- [ ] 返回顺序始终等于输入顺序
- [ ] 用三次数据说明"批次上限不等于服务配额"

参考实现：[practice/](practice/) 下的 `c7_1.py` ~ `c7_5.py`。

---

## ✅ 本次实际运行验收

In [9]:
import sys
from datetime import datetime, timedelta, timezone

expected = {
    "最小工作流与根输出",
    "真实模型流水线与执行顺序",
    "一次通过、预算耗尽与真实模型修改",
    "并发确实发生且不超过上限",
}
assert set(checks) == expected and all(checks.values())
assert len(model_responses) == 4
assert all(
    isinstance(r["tokens"], int) and r["tokens"] > 0 for r in model_responses
)
print(
    "执行时间：",
    datetime.now(timezone(timedelta(hours=8))).isoformat(timespec="seconds"),
)
print(
    "环境：Python",
    sys.version.split()[0],
    "/ google-adk",
    metadata.version("google-adk"),
)
for name in checks:
    print("✅", name)
print("真实 DeepSeek 响应次数：", len(model_responses))
print("服务返回的总 token 使用：", sum(r["tokens"] for r in model_responses))
print("✅ 第八章全部验收通过")

执行时间： 2026-10-19T01:02:29+08:00
环境：Python 3.12.3 / google-adk 2.12.0
✅ 最小工作流与根输出
✅ 真实模型流水线与执行顺序
✅ 一次通过、预算耗尽与真实模型修改
✅ 并发确实发生且不超过上限
真实 DeepSeek 响应次数： 4
服务返回的总 token 使用： 1169
✅ 第八章全部验收通过
